필수 모듈 임포트 및 디바이스 설정

In [2]:
import os
import pandas as pd
from PIL import Image
import tqdm
from google.colab import drive

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
import torchvision.models as models
from sklearn.model_selection import train_test_split
import tqdm

# GPU가 사용 가능하면 cuda, 아니면 cpu로 설정
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


In [3]:
drive.mount('/content/drive')

base_path = '/content/drive/MyDrive/Synapse'
train_csv_path = '/content/drive/MyDrive/Synapse/dataset/train.csv'
test_csv_path =  '/content/drive/MyDrive/Synapse/dataset/test.csv'
train_dir =  '/content/drive/MyDrive/Synapse/dataset/train'
test_dir =  '/content/drive/MyDrive/Synapse/dataset/test'
sample_submission_path =  '/content/drive/MyDrive/Synapse/dataset/sample_submission.csv'


import random
import numpy as np

def set_seed(seed=5):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(5)

train_df = pd.read_csv(train_csv_path)

train_df, val_df = train_test_split(
    train_df,
    test_size=0.2,
    stratify=train_df['label'],
    random_state=42
)


class CustomDataset(Dataset):
    def __init__(self, dataframe, img_dir, transform=None, is_test=False):
        self.df = dataframe.reset_index(drop=True)
        self.img_dir = img_dir
        self.transform = transform
        self.is_test = is_test
        self.file_names = self.df.iloc[:, 0].values

        if not self.is_test:
            self.labels = self.df.iloc[:, 1].values

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        img_name = self.file_names[idx]
        img_path = os.path.join(self.img_dir, img_name)

        # 이미지를 RGB 형태로 불러오기
        image = Image.open(img_path).convert('RGB')

        if self.transform:
            image = self.transform(image)

        if self.is_test:
            return image

        label = self.labels[idx]
        return image, label

Mounted at /content/drive


전이학습용 데이터 전처리 및 DataLoader 설정

In [4]:
# 전이학습(ImageNet 가중치)을 위한 정규화 수치 적용
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

# Dataset 정의 (기존 baseline 구조 유지)
train_dataset = CustomDataset(train_df, train_dir, transform=train_transform, is_test=False)
val_dataset = CustomDataset(val_df, train_dir, transform=train_transform, is_test=False)

test_df = pd.read_csv(test_csv_path)
test_dataset = CustomDataset(test_df, test_dir, transform=train_transform, is_test=True)

# DataLoader 정의
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=16, shuffle=False, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False, num_workers=2)

모델 학습 및 검증 함수 정의

In [5]:
def train_model(model, train_loader, val_loader, optimizer, device="cuda", epochs=10, criterion=None, scheduler=None, save_path=None):
    device = torch.device(device)
    model = model.to(device)
    criterion = criterion or nn.CrossEntropyLoss()

    history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}
    best_val_acc = 0.0

    for epoch in range(epochs):
        # ------------------------
        # 1. Train Step
        # ------------------------
        model.train()
        train_loss, train_correct, train_total = 0.0, 0, 0
        progress_bar = tqdm.tqdm(train_loader, desc=f"Epoch [{epoch + 1}/{epochs}]")

        for images, labels in progress_bar:
            images, labels = images.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            batch_size = labels.size(0)
            train_loss += loss.item() * batch_size
            predictions = outputs.argmax(dim=1)
            train_correct += (predictions == labels).sum().item()
            train_total += batch_size

            progress_bar.set_postfix(train_loss=f"{train_loss/train_total:.4f}", train_acc=f"{train_correct/train_total:.4f}")

        train_loss /= train_total
        train_acc = train_correct / train_total

        # ------------------------
        # 2. Validation Step
        # ------------------------
        model.eval()
        val_loss, val_correct, val_total = 0.0, 0, 0

        with torch.no_grad():
            for images, labels in val_loader:
                images, labels = images.to(device), labels.to(device)
                outputs = model(images)
                loss = criterion(outputs, labels)

                batch_size = labels.size(0)
                val_loss += loss.item() * batch_size
                predictions = outputs.argmax(dim=1)
                val_correct += (predictions == labels).sum().item()
                val_total += batch_size

        val_loss /= val_total
        val_acc = val_correct / val_total

        # 기록 및 출력
        history["train_loss"].append(train_loss)
        history["train_acc"].append(train_acc)
        history["val_loss"].append(val_loss)
        history["val_acc"].append(val_acc)

        print(f"Epoch [{epoch + 1}/{epochs}] | Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f} | Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f}")

        if scheduler is not None:
            scheduler.step()

        # 최고 가중치 저장
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            if save_path:
                torch.save(model.state_dict(), save_path)

    print(f"Best Validation Accuracy: {best_val_acc:.4f}")
    return history

ResNet18 불러오기 및 Classifier 수정

In [6]:
# 1. 사전 학습된 ResNet18 가중치 로드
transfer_model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)

# 2. 마지막 전결합층(fc)의 입력 노드 수 확인
num_ftrs = transfer_model.fc.in_features

# 3. 출력 노드를 10개(클래스 수)로 변경하는 새로운 Linear 레이어 설정
transfer_model.fc = nn.Linear(num_ftrs, 10)

# 4. 모델을 연산 디바이스(GPU/CPU)로 이동
transfer_model = transfer_model.to(device)

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 187MB/s]


옵티마이저 설정 및 전이학습 실행

In [7]:
# 옵티마이저 설정 (학습률은 Fine-tuning에 적합한 1e-4로 설정)
optimizer = torch.optim.Adam(transfer_model.parameters(), lr=1e-4)

history_tf = train_model(
    model=transfer_model,
    train_loader=train_loader,
    val_loader=val_loader,
    optimizer=optimizer,
    device=device,
    epochs=15,
    save_path=base_path + "/resnet18_transfer.pt"
)

Epoch [1/15]: 100%|██████████| 37/37 [02:26<00:00,  3.96s/it, train_acc=0.8097, train_loss=0.7400]


Epoch [1/15] | Train Loss: 0.7400 | Train Acc: 0.8097 | Val Loss: 0.0902 | Val Acc: 0.9862


Epoch [2/15]: 100%|██████████| 37/37 [00:14<00:00,  2.57it/s, train_acc=0.9965, train_loss=0.0497]


Epoch [2/15] | Train Loss: 0.0497 | Train Acc: 0.9965 | Val Loss: 0.0285 | Val Acc: 1.0000


Epoch [3/15]: 100%|██████████| 37/37 [00:14<00:00,  2.52it/s, train_acc=1.0000, train_loss=0.0201]


...

Epoch [13/15] | Train Loss: 0.0165 | Train Acc: 0.9965 | Val Loss: 0.0100 | Val Acc: 1.0000


Epoch [14/15]: 100%|██████████| 37/37 [00:13<00:00,  2.71it/s, train_acc=0.9983, train_loss=0.0186]


Epoch [14/15] | Train Loss: 0.0186 | Train Acc: 0.9983 | Val Loss: 0.0133 | Val Acc: 0.9931


Epoch [15/15]: 100%|██████████| 37/37 [00:12<00:00,  3.02it/s, train_acc=1.0000, train_loss=0.0076]


Epoch [15/15] | Train Loss: 0.0076 | Train Acc: 1.0000 | Val Loss: 0.0149 | Val Acc: 0.9931
Best Validation Accuracy: 1.0000


테스트 데이터 예측 및 제출 파일 생성

In [8]:
from torch.serialization import load

transfer_model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
transfer_model.fc = nn.Linear(transfer_model.fc.in_features, 10)

transfer_model.load_state_dict(torch.load(base_path + "/resnet18_transfer.pt"))
transfer_model = transfer_model.to(device)

def predict_gen_submission_transfer(csv_name, model, test_loader):
    sample_submission = pd.read_csv(sample_submission_path)
    predictions = []
    model.eval()

    with torch.no_grad():
        for images in test_loader:
            images = images.to(device)
            outputs = model(images)
            preds = outputs.argmax(dim=1)
            predictions.extend(preds.cpu().numpy())

    sample_submission.iloc[:, 1] = predictions
    sample_submission.to_csv(base_path + '/' + csv_name + '.csv', index=False)
    print(f'{csv_name}.csv 파일이 구글 드라이브에 성공적으로 저장되었습니다')

In [9]:
predict_gen_submission_transfer("transfer_submmision",transfer_model,test_loader)

transfer_submmision.csv 파일이 구글 드라이브에 성공적으로 저장되었습니다
